# 03 retrieval optimization

**Owner:** Daniel. Original notebook cells: 70, 71.

Run in the shared Databricks session using `run_pipeline.ipynb` or the documented stage runner. This is not a standalone clean-kernel notebook. Saved outputs have been removed for public-repository safety. See `docs/execution_order.md`.

## Fixed 20-Question Eval Set & Retrieval Parameter Experiment

In [ ]:
# Fixed RAG Evaluation Question Set & Retrieval Experiments
# Evaluates chunk size / overlap / top_k before running full LLM grounding eval.

import json
import numpy as np
import pandas as pd

TEAM_VOLUME = "/Volumes/workspace/default/eye_style_team2"
RAG_DATA_DIR = f"{TEAM_VOLUME}/rag"

# Fixed 20-question evaluation set (15 in-corpus + 5 out-of-corpus must-abstain)
RAG_EVAL_QUESTIONS = [
    # --- Scheduling & No-Show Triage (DOC-01, DOC-02) ---
    {
        "question_id": "RAG-01",
        "question": "How long is a standard Comprehensive Eye Exam appointment and how far in advance can routine exams be booked?",
        "expected_source": "DOC-01",
        "expected_section": "Appointment Durations & Booking Windows",
        "insufficient_evidence_expected": False,
        "role": "Front Desk",
    },
    {
        "question_id": "RAG-02",
        "question": "How many days before a visit does front desk perform appointment-readiness review, and can the AI automatically cancel appointments?",
        "expected_source": "DOC-01",
        "expected_section": "Pre-Visit Readiness & Cancellation Policy",
        "insufficient_evidence_expected": False,
        "role": "Front Desk",
    },
    {
        "question_id": "RAG-03",
        "question": "What classification threshold flags an appointment as High priority in the no-show model, and should staff describe the score as a probability?",
        "expected_source": "DOC-02",
        "expected_section": "Interpreting Booking-Time Model Risk Scores",
        "insufficient_evidence_expected": False,
        "role": "Front Desk",
    },
    {
        "question_id": "RAG-04",
        "question": "What outreach cadence should front-desk staff follow for High-priority no-show appointments?",
        "expected_source": "DOC-02",
        "expected_section": "Human-in-the-Loop Outreach Cadence",
        "insufficient_evidence_expected": False,
        "role": "Front Desk",
    },
    # --- Insurance Verification & Payer Rules (DOC-03, DOC-04) ---
    {
        "question_id": "RAG-05",
        "question": "When must front-desk staff verify insurance eligibility before an appointment, and does the capstone AI connect to live VSP portals?",
        "expected_source": "DOC-03",
        "expected_section": "Standard Pre-Visit Verification Timeline",
        "insufficient_evidence_expected": False,
        "role": "Front Desk",
    },
    {
        "question_id": "RAG-06",
        "question": "How should staff handle insurance records marked Needs Review, Pending, or Expired if still unresolved within 24 hours of a visit?",
        "expected_source": "DOC-03",
        "expected_section": "Handling Insurance Exceptions & Escalation",
        "insufficient_evidence_expected": False,
        "role": "Front Desk",
    },
    {
        "question_id": "RAG-07",
        "question": "Should a diabetic retinal evaluation or glaucoma monitoring visit be billed under a routine vision plan or medical insurance?",
        "expected_source": "DOC-04",
        "expected_section": "Vision Plans vs. Medical Insurance Routing",
        "insufficient_evidence_expected": False,
        "role": "Front Desk",
    },
    {
        "question_id": "RAG-08",
        "question": "Can a patient combine full frame material allowances and elective contact lens allowances in the same benefit cycle on standard VSP Choice plans?",
        "expected_source": "DOC-04",
        "expected_section": "Coordination of Benefits & Authorizations",
        "insufficient_evidence_expected": False,
        "role": "Front Desk",
    },
    # --- Optical Orders, Deposits & Contact Lenses (DOC-05, DOC-06, DOC-07) ---
    {
        "question_id": "RAG-09",
        "question": "What are the three core MVP optical order tracking statuses and what does ReadyForDispensing mean?",
        "expected_source": "DOC-05",
        "expected_section": "Optical Order Status Definitions",
        "insufficient_evidence_expected": False,
        "role": "Optician",
    },
    {
        "question_id": "RAG-10",
        "question": "What are the standard lab turnaround times for digital progressives and what should the optician do first if an order is overdue?",
        "expected_source": "DOC-05",
        "expected_section": "Lab Turnaround Times & Overdue Escalation",
        "insufficient_evidence_expected": False,
        "role": "Optician",
    },
    {
        "question_id": "RAG-11",
        "question": "What minimum deposit is required before submitting custom prescription frame orders to an optical lab?",
        "expected_source": "DOC-06",
        "expected_section": "Order Deposits & Pre-Dispense Quality Inspection",
        "insufficient_evidence_expected": False,
        "role": "Optician",
    },
    {
        "question_id": "RAG-12",
        "question": "What is the pickup notification schedule for ReadyForDispensing orders and when are unclaimed orders escalated to the Practice Owner?",
        "expected_source": "DOC-06",
        "expected_section": "Pickup Notification Cadence & Unclaimed Orders",
        "insufficient_evidence_expected": False,
        "role": "Optician",
    },
    {
        "question_id": "RAG-13",
        "question": "How long are standard contact lens prescriptions valid, and why do synthetic contact lens order records omit optical lab fields?",
        "expected_source": "DOC-07",
        "expected_section": "Prescription Validity & Annual Evaluation",
        "insufficient_evidence_expected": False,
        "role": "Optician",
    },
    # --- Onboarding, Governance & Escalation (DOC-08, DOC-09, DOC-10) ---
    {
        "question_id": "RAG-14",
        "question": "Which workflows can an Optician access on the Eye Style AI platform, and which specialist agents are denied to them?",
        "expected_source": "DOC-09",
        "expected_section": "Role-Based Access & Scoped Specialist Agents",
        "insufficient_evidence_expected": False,
        "role": "Owner",
    },
    {
        "question_id": "RAG-15",
        "question": "What should front-desk staff or opticians do if a patient reports sudden vision loss, new flashes and floaters, or chemical eye exposure?",
        "expected_source": "DOC-10",
        "expected_section": "Clinical Symptom & Urgent Medical Escalation",
        "insufficient_evidence_expected": False,
        "role": "Front Desk",
    },
    # --- Insufficient-Evidence / Out-of-Corpus Questions (MUST ABSTAIN) ---
    {
        "question_id": "RAG-16",
        "question": "What is Eye Style Optometry's exact 2026 retail cash price for Zeiss SmartLife progressive lenses with Crizal Sapphire coating?",
        "expected_source": "NONE (Out of corpus)",
        "expected_section": "N/A",
        "insufficient_evidence_expected": True,
        "role": "Optician",
    },
    {
        "question_id": "RAG-17",
        "question": "Which prescription antibiotic eye drops and dosing schedule should we recommend for bacterial conjunctivitis?",
        "expected_source": "NONE (Out of corpus)",
        "expected_section": "N/A",
        "insufficient_evidence_expected": True,
        "role": "Front Desk",
    },
    {
        "question_id": "RAG-18",
        "question": "What is the clinic's employee PTO accrual policy and holiday overtime pay rate for opticians?",
        "expected_source": "NONE (Out of corpus)",
        "expected_section": "N/A",
        "insufficient_evidence_expected": True,
        "role": "Owner",
    },
    {
        "question_id": "RAG-19",
        "question": "What is the current live VSP remaining frame allowance balance for patient Jane Doe today?",
        "expected_source": "NONE (Out of corpus)",
        "expected_section": "N/A",
        "insufficient_evidence_expected": True,
        "role": "Front Desk",
    },
    {
        "question_id": "RAG-20",
        "question": "What third-party collections agency and interest rate does Eye Style use for invoices past 120 days overdue?",
        "expected_source": "NONE (Out of corpus)",
        "expected_section": "N/A",
        "insufficient_evidence_expected": True,
        "role": "Owner",
    },
]

# Persist the fixed evaluation question set before running experiments
with open(f"{RAG_DATA_DIR}/rag_eval_questions_v1.json", "w", encoding="utf-8") as f:
    json.dump(RAG_EVAL_QUESTIONS, f, indent=2)

# Retrieval hyperparameter grid (chunk size, overlap, top_k)
experiment_configs = [
    {"config_id": "EXP-1", "chunk_size": 60,  "overlap": 15, "top_k": 3},
    {"config_id": "EXP-2", "chunk_size": 95,  "overlap": 25, "top_k": 1},
    {"config_id": "EXP-3", "chunk_size": 95,  "overlap": 25, "top_k": 3},
    {"config_id": "EXP-4", "chunk_size": 95,  "overlap": 25, "top_k": 5},
    {"config_id": "EXP-5", "chunk_size": 140, "overlap": 35, "top_k": 3},
]

# Cache indices per (chunk_size, overlap) so we don't re-encode unnecessarily
index_cache = {}
exp_rows = []

answerable_qs = [q for q in RAG_EVAL_QUESTIONS if not q["insufficient_evidence_expected"]]
abstain_qs = [q for q in RAG_EVAL_QUESTIONS if q["insufficient_evidence_expected"]]

for cfg in experiment_configs:
    key = (cfg["chunk_size"], cfg["overlap"])
    if key not in index_cache:
        chunks = build_rag_chunks(
            loaded_docs,
            chunk_size_words=cfg["chunk_size"],
            chunk_overlap_words=cfg["overlap"],
        )
        index_cache[key] = OptometryRAGIndex(chunks)
    idx = index_cache[key]

    doc_hits = 0
    rr_sum = 0.0
    in_top_scores = []
    for q in answerable_qs:
        results = idx.retrieve(q["question"], top_k=cfg["top_k"])
        in_top_scores.append(results[0]["similarity"] if results else 0.0)
        retrieved_doc_ids = [r["doc_id"] for r in results]
        if q["expected_source"] in retrieved_doc_ids:
            doc_hits += 1
            rank = retrieved_doc_ids.index(q["expected_source"]) + 1
            rr_sum += 1.0 / rank

    out_top_scores = [
        idx.retrieve(q["question"], top_k=1)[0]["similarity"]
        for q in abstain_qs
    ]

    exp_rows.append({
        "config_id": cfg["config_id"],
        "chunk_size_words": cfg["chunk_size"],
        "overlap_words": cfg["overlap"],
        "top_k": cfg["top_k"],
        "num_chunks": len(idx.chunks),
        "doc_hit_rate": round(doc_hits / len(answerable_qs), 4),
        "mrr": round(rr_sum / len(answerable_qs), 4),
        "mean_in_corpus_sim": round(float(np.mean(in_top_scores)), 4),
        "mean_out_corpus_sim": round(float(np.mean(out_top_scores)), 4),
        "sim_margin": round(float(np.mean(in_top_scores) - np.mean(out_top_scores)), 4),
    })

rag_experiments_df = pd.DataFrame(exp_rows)
rag_experiments_df.to_csv(f"{RAG_DATA_DIR}/rag_retrieval_experiments.csv", index=False)

print("=== RAG CHUNKING & RETRIEVAL EXPERIMENTS ===")
display(rag_experiments_df)

# Lock winning configuration (prefer highest hit rate & MRR; prefer top_k=3 on ties for multi-section SOP queries)
rag_experiments_df["prefers_multi_chunk"] = (rag_experiments_df["top_k"] == 3).astype(int)

best_exp = rag_experiments_df.sort_values(
    by=["doc_hit_rate", "mrr", "prefers_multi_chunk", "sim_margin"],
    ascending=[False, False, False, False],
).iloc[0]

SELECTED_RAG_CONFIG = {
    "config_id": best_exp["config_id"],
    "chunk_size_words": int(best_exp["chunk_size_words"]),
    "overlap_words": int(best_exp["overlap_words"]),
    "top_k": int(best_exp["top_k"]),
    "backend": index_cache[(int(best_exp["chunk_size_words"]), int(best_exp["overlap_words"]))].backend,
}
rag_index = index_cache[(SELECTED_RAG_CONFIG["chunk_size_words"], SELECTED_RAG_CONFIG["overlap_words"])]
RAG_CONFIG["top_k"] = SELECTED_RAG_CONFIG["top_k"]

print("\nSelected RAG configuration:")
print(json.dumps(SELECTED_RAG_CONFIG, indent=2))